# Part 1 — Train Stack

This notebook provisions the training environment: a short-lived VM that builds a Docker image, trains the PySpark fraud-detection model, and pushes a versioned scoring image to Artifact Registry. It assumes the shared **data stack** from [tutorial_1.ipynb](tutorial_1.ipynb) is already deployed.

## Contents

1. [Train Stack](#10-train-stack)
    - [1.1 Networking](#11-networking)
    - [1.2 Model Preprocessing and Training](#12-model-preprocessing-and-training)
    - [1.3 Saving Image and Model](#13-saving-image-and-model)
    - [1.4 Vulnerability scanning](#14-vulnerability-scanning)
    - [1.5 Inspecting outputs](#15-inspecting-outputs)

← Back to [tutorial_1.ipynb](tutorial_1.ipynb) · → Continue to [tutorial_3.ipynb](tutorial_3.ipynb)

## 1.0 Train Stack
The train stack creates a short-lived private network and a VM. Its process breaks down into 7 steps, shown in figure 1 below:

1. **Boot VM** — Terraform creates the train VM; `startup.sh` begins running.

2. **Build** — `startup.sh` builds the base Docker image (`fraud-detection:base`).

3. **Run** — the base image runs the PySpark training container.

4. **Save model** — training finishes and saves the `PipelineModel`; `startup.sh` builds the derived scoring image (`fraud-scoring:v5`) with the model baked in.

5. **Writes** — the training run also writes `summary.json` locally with the run metrics.

6. **Docker push** — the scoring image is pushed to Artifact Registry in the data stack.

7. **gsutil cp** — `summary.json` is uploaded to the GCS bucket in the data stack.

The VM sits inside a private subnet with no public IP. Two managed gateways provide the only paths in and out: Identity-Aware Proxy (IAP) carries inbound SSH from the operator, and Cloud NAT carries outbound traffic to the public internet used to download dependencies (apt, docker pull, gcloud). The VM never routes to the internet directly in either direction, and no port on it is exposed to the open internet.

The stack is short-lived by design: it exists only long enough to produce the trained model and the scoring image, and can be torn down with `terraform destroy` once both artefacts land in GCS and Artifact Registry. Nothing downstream depends on the VM staying alive.

<p align="center">
  <img src="images/train_pipeline.png" width="600"><br>
  Fig 1. Train Stack Pipeline
</p>

**Startup-script orchestration.** Everything the VM needs to run the pipeline above is baked into its boot-time script. Terraform's `templatefile()` renders `startup.sh` at boot, substituting the Dockerfiles, Python scripts, and data-stack values the script expects:

```hcl
# infra/terraform/train/main.tf (abridged)
metadata_startup_script = templatefile("${path.module}/startup.sh", {
  dockerfile_txt         = file("${path.module}/../../../docker/Dockerfile")
  dockerfile_scoring_txt = file("${path.module}/../../../docker/Dockerfile.scoring")
  fraud_script_py        = file("${path.module}/../../../scripts/fraud_detection_pyspark.py")
  score_script_py        = file("${path.module}/../../../scripts/score_stream.py")
  dataset_bucket_name    = local.dataset_bucket_name
  scoring_image_url      = local.scoring_image_url
  # ...plus a few more fields read from the data stack
})
```

### 1.1 Networking

Figure 2 below shows the network built around the train VM: a private subnet inside a stack-local VPC, with IAP for inbound SSH on one side and Cloud NAT for outbound traffic on the other. The VPC and subnet are created with the train stack and destroyed with it, so no other project resources share them. `private_ip_google_access` is enabled on the subnet so the VM can still reach Google APIs (GCS, Artifact Registry, metadata server).

<p align="center">
  <img src="images/train_network.png" width="600"><br>
  Fig 2. Train stack network
</p>

**IAP SSH:** The only inbound path to the VM. The operator runs `gcloud compute ssh --tunnel-through-iap`, which tunnels through IAP using their `gcloud` identity. The firewall locks port 22 to IAP's range (`35.235.240.0/20`), which is Google's IP range for IAPs:

```hcl
# infra/terraform/train/main.tf (abridged)
resource "google_compute_firewall" "allow_iap_ssh" {
  name    = "${var.instance_name}-allow-iap-ssh"
  network = google_compute_network.train_vpc.name

  allow {
    protocol = "tcp"
    ports    = ["22"]
  }

  source_ranges = ["35.235.240.0/20"]   # IAP only; no public 0.0.0.0/0
  target_tags   = ["ssh-iap"]
}
```

**Cloud NAT:** Cloud NAT gives the VM outbound internet access so the startup script can `apt-get update`, pull the python base image, and reach Google APIs.

### 1.2 Model Preprocessing and Training

#### Preprocessing

The training script runs Spark in local mode on the VM, using all cores as workers. The dataset is read into a DataFrame, cleaned (nulls dropped, duplicates removed, etc.), and cached in memory. A temporary SQL view is registered over the cleaned DataFrame so per-class summary statistics can be computed in plain SQL alongside the DataFrame code.

The cleaned DataFrame is then split 80/20 into train and test sets. Because positives are only ~0.17% of the data, a class weight is attached to each row which up-weights the minority class inside the loss function in order to counteract the class imbalance.

#### Training

The classifier is packaged as a `spark.ml` `Pipeline` so preprocessing is bound to the model itself. A Pipeline is an ordered sequence of stages, each of which is either a *transformer* (stateless, applies a function to a DataFrame) or an *estimator* (stateful, learns parameters from a DataFrame and returns a fitted transformer). Chaining stages this way prevents data leakage: each stage is fit only on the training split and reapplied at inference.

The pipeline defined in [scripts/fraud_detection_pyspark.py](scripts/fraud_detection_pyspark.py) has three stages:

1. **`VectorAssembler`** — a transformer that combines the input columns into a single feature vector.
2. **`StandardScaler`** — an estimator that learns each feature's standard deviation on the training set and rescales features to unit variance.
3. **`LogisticRegression`** — an estimator that fits the binary classifier, using the class weight from the preprocessing step and a raised decision threshold to tighten precision on the fraud class.

The three stages are composed into a single `Pipeline` and fit in one call:

```python
# scripts/fraud_detection_pyspark.py (abridged)
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler    = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True)
lr        = LogisticRegression(featuresCol="features", labelCol="label",
                               weightCol="weight", threshold=args.decision_threshold)

pipeline = Pipeline(stages=[assembler, scaler, lr])
model    = pipeline.fit(train_df)
```

Performance is measured with `BinaryClassificationEvaluator`, reporting AUC-ROC and AUC-PR — the latter is the primary metric given the class imbalance.

#### Evaluation

On the held-out test set the pipeline produces **AUC-ROC ≈ 0.96** and **AUC-PR ≈ 0.71**

### 1.3 Saving Image and Model
After training completes, three artefacts need to be produced: the trained `PipelineModel`, a `summary.json` of run metrics, and a scoring container image for the stream stack.

The `startup.sh` builds `fraud-scoring:v5` on top of the base image, adding the saved `PipelineModel` as a single new layer (see [docker/Dockerfile.scoring](docker/Dockerfile.scoring)):

```dockerfile
# docker/Dockerfile.scoring
ARG BASE_IMAGE
FROM ${BASE_IMAGE}

COPY model /app/model

CMD ["python", "scripts/score_stream.py", "--model-dir", "/app/model"]
```

Because only that `COPY model` layer differs from the base, the push uploads just the model. The Python environment and scoring scripts are already in the registry.

In parallel, `summary.json` is copied to the data stack's GCS bucket via `gsutil cp`. Once both land, the train VM's work is done and it can be destroyed.

> **Deploy now.** To spin the train stack up as you read, run `terraform init && terraform apply` in `infra/terraform/train/`. The VM boots, trains the model, and pushes the scoring image unattended, then can be destroyed with `terraform destroy` once both artefacts land.

### 1.4 Vulnerability scanning

Once the scoring image was in Artifact Registry, we ran the following command once to list the vulnerabilities on it:

```bash
gcloud artifacts docker images list \
  europe-west2-docker.pkg.dev/<project_id>/fraud-detection-images/fraud-scoring \
  --show-occurrences --occurrence-filter='kind="VULNERABILITY"'
```

This returned **2 CRITICAL, 32 HIGH, 35 MEDIUM, 28 LOW, 26 MINIMAL** findings, all in the Debian trixie base layer (`glibc`, `tar`, `nss`, `sqlite3`, `shadow`, `systemd`), none in application code or Python dependencies. Two flags on every finding make them non-actionable today:

- `effectiveSeverity: MINIMAL` — Google's contextual re-score: the vulnerable code paths are not reachable in this image's runtime context.
- `fixedVersion.kind: MAXIMUM` — no upstream fix exists in Debian, so `apt-get upgrade` would change nothing.

### 1.5 Inspecting outputs

The README's [Inspecting each phase](README.md#inspecting-each-phase) section lists the full set of `gcloud` commands. The two most useful for Part 1:

```bash
# Watch the VM's startup log live (build → train → push)
gcloud compute ssh fraud-train-vm --zone europe-west2-a --tunnel-through-iap \
  --command 'sudo tail -f /var/log/startup.log'

# Pretty-print training metrics straight from GCS — no SSH
gcloud storage cat gs://<bucket>/summaries/summary.json | jq
```

The first SSH happens through IAP because the VM has no public IP. `--tunnel-through-iap` is what makes `gcloud ssh` work despite the firewall locking port 22 to `35.235.240.0/20`.

---

→ Continue to [Part 2 — Stream Stack](tutorial_3.ipynb).